# Understanding Data Types in Python

Effective data-driven science and computation requires understanding how data is stored and manipulated.
This section outlines and contrasts how arrays of data are handled in the Python language itself, and how NumPy improves on this.
Understanding this difference is fundamental to understanding much of the python.

Users of Python are often drawn-in by its ease of use, one piece of which is dynamic typing.
While a statically-typed language like C or Java requires each variable to be explicitly declared, a dynamically-typed language like Python skips this specification. For example, in C you might specify a particular operation as follows:

```C
/* C code */
int result = 0;
for(int i=0; i<100; i++){
    result += i;
}
```

While in Python the equivalent operation could be written this way:

```python
# Python code
result = 0
for i in range(100):
    result += i
```

Notice the main difference: in C, the data types of each variable are explicitly declared, while in Python the types are dynamically inferred. This means, for example, that we can assign any kind of data to any variable:

```python
# Python code
x = 4
x = "four"
```

Here we've switched the contents of ``x`` from an integer to a string. The same thing in C would lead (depending on compiler settings) to a compilation error or other unintented consequences:

```C
/* C code */
int x = 4;
x = "four";  // FAILS
```

This sort of flexibility is one piece that makes Python and other dynamically-typed languages convenient and easy to use.
Understanding *how* this works is an important piece of learning to analyze data efficiently and effectively with Python.
But what this type-flexibility also points to is the fact that Python variables are more than just their value; they also contain extra information about the type of the value. We'll explore this more in the sections that follow.

## A Python Integer Is More Than Just an Integer

The standard Python implementation is written in C.
This means that every Python object is simply a cleverly-disguised C structure, which contains not only its value, but other information as well. For example, when we define an integer in Python, such as ``x = 10000``, ``x`` is not just a "raw" integer. It's actually a pointer to a compound C structure, which contains several values.
Looking through the Python 3.4 source code, we find that the integer (long) type definition effectively looks like this (once the C macros are expanded):

```C
struct _longobject {
    long ob_refcnt;
    PyTypeObject *ob_type;
    size_t ob_size;
    long ob_digit[1];
};
```

A single integer in Python 3.4 actually contains four pieces:

- ``ob_refcnt``, a reference count that helps Python silently handle memory allocation and deallocation
- ``ob_type``, which encodes the type of the variable
- ``ob_size``, which specifies the size of the following data members
- ``ob_digit``, which contains the actual integer value that we expect the Python variable to represent.

This means that there is some overhead in storing an integer in Python as compared to an integer in a compiled language like C, as illustrated in the following figure:

![Integer Memory Layout](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/figures/cint_vs_pyint.png?raw=1)

Here ``PyObject_HEAD`` is the part of the structure containing the reference count, type code, and other pieces mentioned before.

Notice the difference here: a C integer is essentially a label for a position in memory whose bytes encode an integer value.
A Python integer is a pointer to a position in memory containing all the Python object information, including the bytes that contain the integer value.
This extra information in the Python integer structure is what allows Python to be coded so freely and dynamically.
All this additional information in Python types comes at a cost, however, which becomes especially apparent in structures that combine many of these objects.

## A Python List Is More Than Just a List

Let's consider now what happens when we use a Python data structure that holds many Python objects.
The standard mutable multi-element container in Python is the list.
We can create a list of integers as follows:

In [1]:
L = list(range(10))
L

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [2]:
type(L)

list

In [5]:
type(L[0])

int

In [8]:
a = str(L)
type(a[0])
#a[0]

str

### How to convert each item present in the list as string to extract the first element.

Or, similarly, a list of strings:

In [9]:
x = []

for c in L:
    num = str(c)
    x.append(num)
    
x

'0'

In [15]:
num1 = [str(c) for c in L]
num1

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [17]:
[type(c) for c in num1]

[str, str, str, str, str, str, str, str, str, str]

### List Comprehension

List comprehension is a concise, readable way to create lists in Python. It replaces traditional ``for`` loops used for building lists with a single line of code.

In [18]:
L2 = [str(Li) for Li in L]
L2

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [19]:
type(L2[0])

str

## Example 2

In [13]:
#myList=[]

for i in range(10):
    if i%2 == 0:     
        myList.append(i)
    
myList

[0, 2, 4, 6, 8]

In [14]:
myList = [i for i in range(10) if i%2 == 0]
myList

[0, 2, 4, 6, 8]

## Example 3

In [ ]:
# Convert even numbers to 'Even' and odd numbers to 'Odd'
labels = ["Even" if x % 2 == 0 else "Odd" for x in range(5)]
# Result: ['Even', 'Odd', 'Even', 'Odd', 'Even']

Because of Python's dynamic typing, we can even create heterogeneous lists:

In [25]:
L3 = [True, "2", 3.0, 4]
[type(item) for item in L3]

[bool, str, float, int]

But this flexibility comes at a cost: to allow these flexible types, each item in the list must contain its own type info, reference count, and other information–that is, each item is a complete Python object.
In the special case that all variables are of the same type, much of this information is redundant: it can be much more efficient to store data in a fixed-type array.
The difference between a dynamic-type list and a fixed-type (NumPy-style) array is illustrated in the following figure:

![Array Memory Layout](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/figures/array_vs_list.png?raw=1)

At the implementation level, the array essentially contains a single pointer to one contiguous block of data.
The Python list, on the other hand, contains a pointer to a block of pointers, each of which in turn points to a full Python object like the Python integer we saw earlier.
Again, the advantage of the list is flexibility: because each list element is a full structure containing both data and type information, the list can be filled with data of any desired type.
Fixed-type NumPy-style arrays lack this flexibility, but are much more efficient for storing and manipulating data.

## Fixed-Type Arrays in Python

Python offers several different options for storing data in efficient, fixed-type data buffers.
The built-in ``array`` module (available since Python 3.3) can be used to create dense arrays of a uniform type:

In [26]:
import array as arr
L = list(range(10))

# creating an array with integer type
A = arr.array('i', L)
print(A)

# creating an array with float type
B = arr.array('d', L)
print(B)

array('i', [0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
array('d', [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])


Here ``'i'`` is a type code indicating the contents are integers.

Much more useful, however, is the ``ndarray`` object of the NumPy package.
While Python's ``array`` object provides efficient storage of array-based data, NumPy adds to this efficient *operations* on that data.
We will explore these operations in later sections; here we'll demonstrate several ways of creating a NumPy array.

We'll start with the standard NumPy import, under the alias ``np``:

In [28]:
import numpy
numpy.__version__

'1.26.2'

In [29]:
import numpy as np

In [30]:
np?

## Creating Arrays from Python Lists

First, we can use ``np.array`` to create arrays from Python lists:

In [31]:
# integer array:
L = list(range(10))
print(L)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [32]:
np.array(L)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

Remember that unlike Python lists, NumPy is constrained to arrays that all contain the same type.
If types do not match, NumPy will upcast if possible (here, integers are up-cast to floating point):

In [33]:
np.array((3.14, 2, 3, 4))

array([3.14, 2.  , 3.  , 4.  ])

In [34]:
np.array([True, "2", 3.0, 4])

array(['True', '2', '3.0', '4'], dtype='<U32')

If we want to explicitly set the data type of the resulting array, we can use the ``dtype`` keyword:

In [35]:
np.array([1, 2, 3, 4], dtype='U32')

array(['1', '2', '3', '4'], dtype='<U32')

In [37]:
a = np.array([1.7, 2, 3, 9])
new_a = a.astype(int)
print(new_a)
print(new_a.dtype)

[1 2 3 9]
int64


Finally, unlike Python lists, NumPy arrays can explicitly be multi-dimensional; here's one way of initializing a multidimensional array using a list of lists:

Let's create a multiple ranges to generate multi-dimensional numbers

In [38]:
[range(i, i+3) for i in [2, 4, 6]]

[range(2, 5), range(4, 7), range(6, 9)]

In [39]:
# nested lists result in multi-dimensional arrays
A = np.array([range(i, i + 3) for i in [2, 4, 6]])
A

array([[2, 3, 4],
       [4, 5, 6],
       [6, 7, 8]])

In [40]:
A[2][1]

7

The inner lists are treated as rows of the resulting two-dimensional array.

In [41]:
A.shape

(3, 3)

In [42]:
column, row = 3, 5
array2D = np.array([[i for i in range(row)] for i in range(column)])
array2D

array([[0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4]])

In [43]:
array2D.shape

(3, 5)